# 6 Parallelisierung und Persistenz

# 6.2 Modelle in Python

| Modell | Speicher | Parallelität | Einsatzgebiet |
| --- | --- | --- | --- |
| Prozesse | getrennt | echte CPU-Parallelität | rechenintensive Aufgaben |
| Threads | gemeinsam | eingeschränkt (GIL) | I/O-lastige Aufgaben |
| Coroutines | gemeinsam | kein echter Parallelismus | viele wartende Aktivitäten |

# 6.3 Parallele Ausführung

## threading.Thread (11)

In [ ]:
import random
import threading
import time

# Simuliere Arbeit mit zufälliger Dauer
def random_sleep(t_max):
    time.sleep(random.uniform(t_max/2, t_max))

def worker(n):
    print(f"Thread {n} läuft")
    random_sleep(0.0001)
    print(f"Thread {n} ist fertig")

threads = [
    threading.Thread(target=worker, args=(i,))
    for i in range(5)
]

print("Alle Threads erstellt, starte sie jetzt...")
for t in threads:
    t.start()
print("Alle Threads gestartet, warte auf ihre Fertigstellung...")
for t in threads:
    t.join()
print("Alle Threads sind fertig.")

## concurrent.futures (12)

In [ ]:
from concurrent.futures import as_completed, ThreadPoolExecutor

def worker(n):
    random_sleep(0.0001)
    return 10 * n

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(worker, i) for i in range(10)]

    for f in as_completed(futures):
        print(f.result())

In [ ]:
from concurrent.futures import ProcessPoolExecutor
# Funktion muss in einer separaten Datei liegen, damit sie von mehreren Prozessen importiert werden kann
from utils_06 import worker

with ProcessPoolExecutor(max_workers=2) as executor:
    results = list(executor.map(worker, [0, 1, 2], ["Hallo", "Welt", "Test"]))

print(results)

## asyncio (13)

In [ ]:
import asyncio

async def fetch(url):
    print(f"{url} gestartet")
    await asyncio.sleep(random.uniform(0, 0.1))
    print(f"{url} ist fertig")
    return f"Ergebnis von {url}"

async def main():
    results = await asyncio.gather(fetch("example.com"), fetch("foobar.de"))
    for r in results:
        print(r)

await main()

## Übungen zu 6.3

### Aufgabe 1 - Wartezeiten simulieren

1. Schreibe eine Funktion `simulate_io(name, delay)`, die kurz wartet und dann einen Text zurückgibt.
2. Führe drei Aufrufe parallel mit einem `ThreadPoolExecutor` aus.
3. Formuliere dieselbe Idee zusätzlich mit `asyncio.gather()`.

# 6.4 Synchronisation

## Konkurrierende Zugriffe (15)

In [ ]:
import threading

class Counter:
    def __init__(self):
        self.counter = 0

    def increase(self):
        for _ in range(1000):
            random_sleep(0.00001)  # Simuliere Arbeit
            value = self.counter
            random_sleep(0.00001)  # Simuliere Arbeit
            self.counter = value + 1

counter = Counter()
threads = [threading.Thread(target=counter.increase) for _ in range(2)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"Erwartet: 2000, Erhalten: {counter.counter}")

## Lock (16)

In [ ]:
import threading


class Counter:
    def __init__(self):
        self.counter = 0
        self.lock = threading.Lock()

    def increase(self):
        for _ in range(1000):
            with self.lock:
                random_sleep(0.00001)  # Simuliere Arbeit
                value = self.counter
                random_sleep(0.00001)  # Simuliere Arbeit
                self.counter = value + 1

counter = Counter()
threads = [threading.Thread(target=counter.increase) for _ in range(2)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"Erwartet: 2000, Erhalten: {counter.counter}")

## Event (17)

In [ ]:
import threading
import time

event = threading.Event()

def waiter():
    print("Waiter wartet auf Signal...")
    event.wait()
    print("Waiter hat Signal erhalten")

def setter():
    time.sleep(2)
    print("Setter setzt Event")
    event.set()

t1 = threading.Thread(target=waiter)
t2 = threading.Thread(target=setter)
t1.start()
t2.start()
t1.join()
t2.join()

## Semaphore (18)

In [ ]:
import threading

semaphore = threading.Semaphore(2)  # maximal 2 gleichzeitig

def access(name):
    with semaphore:
        print(f"{name} hat Zugriff")
        random_sleep(1)
        print(f"{name} gibt frei")

threads = [threading.Thread(target=access, args=(f"T{i}",)) for i in range(5)]
for t in threads:
    t.start()
for t in threads:
    t.join()

## Queue (19)

In [ ]:
import queue
import threading

q = queue.Queue()

def producer():
    for item in range(5):
        random_sleep(0.1)  # Simuliere Arbeit
        q.put(item)
        print(f"Erstellt: {item} ")
    q.put(None)  # Sentinel value

def consumer():
    while True:
        item = q.get()
        if item is None:
            break
        print(f"Genutzt: {item}")

t1 = threading.Thread(target=producer)
t2 = threading.Thread(target=consumer)
t1.start()
t2.start()
t1.join()
t2.join()

## Übungen zu 6.4

### Aufgabe 1 - Synchronisationswerkzeuge auswählen

Es geht darum, ein kleines Verarbeitungssystem zu entwerfen:

1. Vier Worker-Threads sollen gleichzeitig bereitstehen, aber erst nach einem gemeinsamen Startsignal loslaufen.
2. Während der Verarbeitung dürfen höchstens zwei Worker gleichzeitig auf eine knappe Ressource zugreifen, z.B. eine Datenbank.
3. Fertige Ergebnisse sollen an eine Sammelstelle übergeben und dort zentral ausgegeben werden.
4. Entscheide erst, welche Werkzeuge benötigt werden: `Lock`, `Event`, `Semaphore`, `Queue` oder eine Kombination davon.

# 6.5 Typische Probleme

## Deadlocks (21)

In [ ]:
import threading
import time

lock_1 = threading.Lock()
lock_2 = threading.Lock()

def thread_A():
    time.sleep(0.1)
    with lock_1:
        time.sleep(0.1)
        with lock_2:
            print("Thread A: hält 1 und 2")

def thread_B():
    time.sleep(0.1)
    with lock_2:
        time.sleep(0.1)
        with lock_1:
            print("Thread B: hält 1 und 2")

t1 = threading.Thread(target=thread_A)
t2 = threading.Thread(target=thread_B)
t1.start()
t2.start()
t1.join()
t2.join()

# 6.6 Serialisierung

## Pickle (27)

In [ ]:
import pickle
import os

class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __str__(self):
        return f"Point({self.x}, {self.y})"

original_point = Point(3, 7)

filename = "point.pkl"
with open(filename, "wb") as f:
    pickle.dump(original_point, f)

with open(filename, "rb") as f:
    restored_point = pickle.load(f)

print("Wiederhergestellt:", restored_point)
os.unlink(filename)

## Übungen zu 6.6

### Aufgabe 1 - Serialisieren mit JSON

1. Serialisiere ein `dict` mit Namen und jeweiliger Note in JSON und lese es wieder ein.